Probability that a pixel burns,given its landcover type

In [2]:
import geopandas as gpd
import pandas as pd


In [53]:
glimpse_df = pd.read_parquet("../data/new_processed_progressions/ua_clusterid/ua_51283.parquet")
glimpse_df.columns
glimpse_df.head()

,K_FireID,CLUSTERID,K_UniqueID,DATE,x,y,lc_class,used,in_landscape,area_factor_m2_per_pixel,lc_class_name
0,191,51283,50,20230501.0,-106.194476,53.048527,9,0,1,538.630722,Forested Mineral Wetland
1,191,51283,50,20230501.0,-106.194207,53.048527,9,0,1,538.630722,Forested Mineral Wetland
2,191,51283,50,20230501.0,-106.193937,53.048527,9,0,1,538.630722,Forested Mineral Wetland
3,191,51283,50,20230501.0,-106.193668,53.048527,9,0,1,538.630722,Forested Mineral Wetland
4,191,51283,50,20230501.0,-106.193398,53.048527,9,0,1,538.630722,Forested Mineral Wetland


In [54]:
import polars as pl

lf = pl.scan_parquet("../data/new_processed_progressions/ua_clusterid/*.parquet")
df = lf.collect()
df.write_parquet("../data/xgboost_selectivity/ua_pixels.parquet")

KeyboardInterrupt: 

In [55]:
df.shape[0]

1345292748

look at the full df and get the predictor columns and the K_UniqueID

In [56]:
cov_df = pd.read_parquet("../data/full_dataset/fire_selectivity_full_dataset_20260323.parquet")
cov_df.columns

Index(['K_FireID', 'CLUSTERID', 'K_UniqueID', 'DATE', 'lc_class_name',
       'lc_class', 'burned_area', 'available_area', 'bu', 'av', 'bu_area',
       'av_area', 'bp', 'jacobs', 'Chesson', 'region', 'NFIREID', 'Date',
       'DC.mean', 'DMC.mean', 'FWI.mean', 'BUI.mean', 'ISI.mean',
       'DC.mean_num', 'ISI.mean_num', 'BUI.mean_num', 'FWI.mean_num', 'DC_bin',
       'ISI_bin', 'BUI_bin', 'FWI_bin', 'DC.mean_num_bin',
       'DC.mean_num_Quartile_Label', 'ISI.mean_num_bin',
       'ISI.mean_num_Quartile_Label', 'BUI.mean_num_bin',
       'BUI.mean_num_Quartile_Label', 'FWI.mean_num_bin',
       'FWI.mean_num_Quartile_Label', 'canopy_bin', 'class_bin', 'season_bin'],
      dtype='str')

Get the ecozones

In [57]:
import yaml
with open("../config.yml", "r") as stream:
    config = yaml.safe_load(stream)

print(config)

{'default': {'dnbr_path': 'G:/Fire_Selectivity/NickPelletier - do not delete/dNBR rasters/', 'peatland_path': 'E:/Jack/data/peatland_fire_selectivity/Peat_Canopy_2023_11_06.tif', 'progression_path': 'G:/Fire_Selectivity/NickPelletier - do not delete/fire polygons 2023/landscape_processed_polygons_km_oct18.shp', 'worker_log': 'logs/worker_log.txt', 'error_log': 'logs/error_log.txt', 'proj_dir': '!expr getwd()', 'processing_logs': 'logs/fire_processing_logs/', 'used_available': 'data/used_available_per_fire/', 'combined_output': 'data/ua_fire_combined/', 'full_dataset_output': 'data/full_dataset/'}, 'python': {'full_dataset_output': 'data/full_dataset/', 'lc_column': 'Lc_class', 'selectivity_output': 'data/new_processed_progressions/selectivity_results/', 'fwi_input': 'data/FWI_metrics_2023_fires_all_dates_by_progression_Oct17KM_NEWclipped.csv', 'progression': 'data/fire_progression_polygons/landscape_processed_polygons_km_oct18.shp', 'progression_2': 'data/progression_NBAC_km_oct2024/',

In [58]:
from pathlib import Path
progs = Path(config["python"]["progression"])
proj_root = Path.cwd().parent
progs_path = proj_root / progs
progs_path = progs_path.resolve()

progs = gpd.read_file(progs_path)
progs.head()

# get centroids for each fire
if progs.crs.is_geographic:
    progs = progs.to_crs(progs.estimate_utm_crs())

progs["centroid"] = progs.geometry.centroid

progs.head()

,Join_Count,TARGET_FID,Join_Cou_1,TARGET_F_1,CLUSTERID,DATE,AREA,C_AREA,FWI,CONSIS_ID,K_FireID,NFIREID,K_UniqueID,UpdateArea,geometry,centroid
0,1.0,7.0,1.0,5.0,64544.0,20230706.0,1115.6800,4832.900,15,61445.0,265.0,191.0,6.0,765.190151,"MULTIPOLYGON (((6598118.968 2031715.731, 65981...",POINT (6603822.192 2037459.147)
1,1.0,9.0,1.0,7.0,55136.0,20230531.0,10985.1000,473748.000,14,51991.0,425.0,349.0,8.0,7873.413049,"MULTIPOLYGON (((5377765.191 2307852.258, 53777...",POINT (5325302.354 2317242.432)
2,1.0,11.0,1.0,9.0,78344.0,20230803.0,90.6167,1684.270,20,74929.0,856.0,2105.0,10.0,65.784296,"POLYGON ((4379939.048 3866443.535, 4379852.942...",POINT (4379310.711 3869441.879)
3,1.0,14.0,1.0,11.0,55132.0,20230531.0,3442.3400,19554.800,13,54384.0,8.0,582.0,12.0,3021.379481,"MULTIPOLYGON (((8307453.096 1271250.773, 83074...",POINT (8321225.071 1273347.677)
4,1.0,16.0,1.0,13.0,51245.0,20230429.0,284.7510,284.751,25,51245.0,176.0,658.0,14.0,255.524219,"POLYGON ((5265718.089 1940985.447, 5265701.608...",POINT (5263797.221 1944140.847)


In [70]:
ecozone =  gpd.read_file("../data/Ecozones_of_Canada.geojson")
ecozone.head()

,FID,AREA,PERIMETER,ZONE_,ZONE_ID,ECOZONE,ZONE_NAME,ZONE_NOM,geometry
0,1,756.53904,488.52793,2,1,2,Northern Arctic,Haut-Arctique,"POLYGON ((-66.61827 58.9452, -66.50138 58.9729..."
1,2,18.74491,99.14604,3,2,1,Arctic Cordillera,CordillCre arctique,"POLYGON ((-70.95229 82.9057, -71.01337 82.8968..."
2,3,11.39239,58.28926,4,3,1,Arctic Cordillera,CordillCre arctique,"POLYGON ((-80.02043 80.41315, -80.01211 80.396..."
3,4,5.56531,41.07040,5,4,1,Arctic Cordillera,CordillCre arctique,"POLYGON ((-94.02854 80.23635, -93.98545 80.235..."
4,5,60.90420,129.38485,6,5,1,Arctic Cordillera,CordillCre arctique,"POLYGON ((-81.458 76.6817, -81.44023 76.68079,..."


In [75]:
progs_filt = progs[["K_UniqueID", "NFIREID",  "centroid", "geometry"]]
ecozone_filt = ecozone[["ZONE_NAME", "geometry"]]

if progs_filt.crs != ecozone_filt.crs:
    ecozone_filt = ecozone_filt.to_crs(progs_filt.crs)

ecozones = gpd.sjoin(
    progs_filt,
    ecozone_filt,
    how="left",
    predicate="intersects"
)

ecozones.shape[0]

7173

In [76]:
# check that there are not nulls

ecozones["ZONE_NAME"].isna().mean()


np.float64(0.0)

In [77]:

missing = ecozones.loc[ecozones["ZONE_NAME"].isna(), "K_UniqueID"]
missing.head()


Series([], Name: K_UniqueID, dtype: float64)

In [78]:
# left join covariate df to ecozones by K_UniqueID

covariates = pd.merge(
    cov_df,
    ecozones[["K_UniqueID", "ZONE_NAME"]],
    how="left",
    on="K_UniqueID"
)
covariates.columns


Index(['K_FireID', 'CLUSTERID', 'K_UniqueID', 'DATE', 'lc_class_name',
       'lc_class', 'burned_area', 'available_area', 'bu', 'av', 'bu_area',
       'av_area', 'bp', 'jacobs', 'Chesson', 'region', 'NFIREID', 'Date',
       'DC.mean', 'DMC.mean', 'FWI.mean', 'BUI.mean', 'ISI.mean',
       'DC.mean_num', 'ISI.mean_num', 'BUI.mean_num', 'FWI.mean_num', 'DC_bin',
       'ISI_bin', 'BUI_bin', 'FWI_bin', 'DC.mean_num_bin',
       'DC.mean_num_Quartile_Label', 'ISI.mean_num_bin',
       'ISI.mean_num_Quartile_Label', 'BUI.mean_num_bin',
       'BUI.mean_num_Quartile_Label', 'FWI.mean_num_bin',
       'FWI.mean_num_Quartile_Label', 'canopy_bin', 'class_bin', 'season_bin',
       'ZONE_NAME'],
      dtype='str')

In [79]:
# get onlye required columns
cols_to_drop = [
    "CLUSTERID",
    "K_FireID",
    "lc_class",
    "lc_class_name",
    "burned_area",
    "available_area",
    'bu', 'av', 'bu_area',
       'av_area', 'bp', 'jacobs', 'Chesson','NFIREID', 'Date','DC.mean_num', 'ISI.mean_num', 'BUI.mean_num', 'FWI.mean_num' , 'DC.mean_num_Quartile_Label', 'ISI.mean_num_bin',
       'ISI.mean_num_Quartile_Label', 'BUI.mean_num_bin',
       'BUI.mean_num_Quartile_Label', 'FWI.mean_num_bin',
       'FWI.mean_num_Quartile_Label', 'DC.mean_num_bin'
]

covariates = covariates.drop(columns=cols_to_drop)
covariates.head()

,K_UniqueID,DATE,region,DC.mean,DMC.mean,FWI.mean,BUI.mean,ISI.mean,DC_bin,ISI_bin,BUI_bin,FWI_bin,canopy_bin,class_bin,season_bin,ZONE_NAME
0,962,20230608.0,east,124.016,49.501,11.09,49.5536439539328,4.0213981627536,low,moderate,high,high,open,upland,Early,Boreal Shield
1,962,20230608.0,east,124.016,49.501,11.09,49.5536439539328,4.0213981627536,low,moderate,high,high,open,upland,Early,Boreal Shield
2,962,20230608.0,east,124.016,49.501,11.09,49.5536439539328,4.0213981627536,low,moderate,high,high,open,upland,Early,Boreal Shield
3,5508,20230624.0,east,277.553673913043,33.9640652173913,9.6845652173913,52.0153547723157,3.30121277368362,high,moderate,high,moderate,forested,upland,Late,Boreal Shield
4,5508,20230624.0,east,277.553673913043,33.9640652173913,9.6845652173913,52.0153547723157,3.30121277368362,high,moderate,high,moderate,forested,upland,Late,Boreal Shield


In [63]:
covariates.shape[0]

24954

In [80]:
# drop duplicate K_UniqueID
covariates = covariates.drop_duplicates(subset =["K_UniqueID"])
covariates.head()

,K_UniqueID,DATE,region,DC.mean,DMC.mean,FWI.mean,BUI.mean,ISI.mean,DC_bin,ISI_bin,BUI_bin,FWI_bin,canopy_bin,class_bin,season_bin,ZONE_NAME
0,962,20230608.0,east,124.016,49.501,11.09,49.5536439539328,4.0213981627536,low,moderate,high,high,open,upland,Early,Boreal Shield
3,5508,20230624.0,east,277.553673913043,33.9640652173913,9.6845652173913,52.0153547723157,3.30121277368362,high,moderate,high,moderate,forested,upland,Late,Boreal Shield
6,553,20230706.0,west,546.013537037037,60.3714074074074,19.7003703703704,94.5918325130038,5.30904482044867,extreme,high,extreme,high,open,upland,Late,Taiga Shield
12,6963,20230807.0,west,599.107341637011,52.1993131672598,16.1128113879004,85.6913087375831,4.3482927506885,extreme,moderate,extreme,high,open,poor fen,Late,Taiga Cordillera
15,2060,20230606.0,east,184.811625,29.28875,8.70125,41.9491801984716,3.44000532480001,moderate,moderate,high,moderate,forested,upland,Early,Boreal Shield


In [81]:
# covariates to parquet
covariates.to_parquet("../data/covariates/covariates.parquet")

Combine frames with lazy loading

In [82]:
cov_lazy = pl.scan_parquet("../data/covariates/covariates.parquet")
df_lazy = pl.scan_parquet("../data/xgboost_selectivity/ua_pixels.parquet")

In [83]:

pl.Config.set_streaming_chunk_size(50_000)
pl.Config.set_tbl_rows(10)


polars.config.Config

In [84]:
out = df_lazy.join(cov_lazy,on="K_UniqueID",how="left")




look at number of fires per ecozone

In [85]:

counts_cov = (
    cov_lazy
    .group_by("ZONE_NAME")
    .count()
)

counts_df = counts_cov.collect()
counts_df


/var/folders/dz/b20t46w554n5xhsppthz6vsw0000gn/T/ipykernel_62116/1717882057.py:4: DeprecationWarning: `count` was renamed; use `len` instead
  .count()


ZONE_NAME,count
str,u32
"""Taiga Plain""",1755
"""Montane Cordillera""",3
"""Taiga Cordillera""",283
"""Boreal Cordillera""",46
"""Hudson Plain""",318
…,…
"""Southern Arctic""",17
"""Atlantic Maritime""",11
"""Prairie""",4


In [86]:
# plit out into different dfs based on ZONE_NAME

zones = (
    out
    .select("ZONE_NAME")
    .unique()
    .drop_nulls()
    .collect()
    .to_series()
    .to_list()
)


In [88]:
output_dir = Path("../data/by_zone")
output_dir.mkdir(parents=True, exist_ok=True)


for zone in zones:
    safe_zone = (
        zone
        .lower()            # no capitals
        .replace(" ", "_")  # spaces → underscores
    )

    (
        out
        .filter(pl.col("ZONE_NAME") == zone)
        .sink_parquet(output_dir / f"{safe_zone}.parquet")
    )



# XGBoost Fire Selectivity Pipeline — Hudson Plain

This notebook implements a complete analysis pipeline to test the hypothesis
that upland pixels burn at higher rates than peatland classes.

**Sections:**
1. Feature preparation
2. Spatial block cross-validation (KMeans on coordinates)
3. Bayesian hyperparameter tuning (Optuna)
4. Out-of-fold XGBoost predictions
5. Calibration check
6. Log odds selectivity with bootstrap CIs
7. SHAP feature importance & landcover deep-dive

In [ ]:
# ─── Data Loading ─────────────────────────────────────────────────────────────
# Load the full Hudson Plain pixel table (~60 M rows).
# ASSUMPTION: The parquet file is already written by the data-prep cells above.
# ASSUMPTION: The dataframe is named `hudson` with at minimum these columns:
#   used          (int8)   — binary burn label: 1 = burned, 0 = unburned
#   lc_class_name (str)    — human-readable landcover class name
#   x, y          (float)  — geographic coordinates (longitude, latitude, WGS84)
#   DC.mean, DMC.mean, FWI.mean, BUI.mean, ISI.mean (str → float) — fire weather
# ASSUMPTION: Coordinates are in WGS84 lon/lat. KMeans spatial blocking
#   uses Euclidean distance in lon/lat space, which underestimates east-west
#   distances at high latitudes (~50–57°N). Project to UTM for exact blocking.

import polars as pl
from pathlib import Path

DATA_PATH = Path("../data/by_zone/hudson_plain.parquet")
hudson = pl.scan_parquet(DATA_PATH).collect()

n_total  = hudson.shape[0]
burn_rate = hudson["used"].cast(pl.Float64).mean()
print(f"Dataset : {n_total:>12,} pixels")
print(f"Burned  : {int(hudson['used'].cast(pl.Int64).sum()):>12,} pixels")
print(f"Unburned: {n_total - int(hudson['used'].cast(pl.Int64).sum()):>12,} pixels")
print(f"Burn rate: {burn_rate:.3%}")
print(f"Columns  : {hudson.columns}")

## 1 · Imports & Configuration

In [ ]:
import warnings
import numpy as np
import pandas as pd
import polars as pl
import xgboost as xgb
import shap
import optuna
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path
from sklearn.cluster import KMeans
from sklearn.calibration import calibration_curve
from sklearn.metrics import brier_score_loss, roc_auc_score

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)

# ─── Global configuration ──────────────────────────────────────────────────────
RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)

# Columns in the raw dataset
TARGET_COL   = "used"           # binary burn label
LC_COL       = "lc_class_name"  # categorical landcover feature
COORD_COLS   = ["x", "y"]       # geographic coordinates (lon/lat, WGS84)
WEATHER_COLS = ["DC.mean", "DMC.mean", "FWI.mean", "BUI.mean", "ISI.mean"]
# ASSUMPTION: No slope/aspect/moisture index is present in this dataset.
# The only continuous covariates are fire-weather indices (stored as strings).
# If additional covariates are added, append them to WEATHER_COLS.

FEATURE_COLS = [LC_COL] + WEATHER_COLS + COORD_COLS

# Classes excluded from selectivity analysis
# ASSUMPTION: Water (two codes) and unclassified (None / code 0) are excluded
# because they are not relevant to the peatland-vs-upland comparison.
EXCLUDE_CLASSES = {"Water", None}

# Classes treated as "upland" (the secondary reference for log odds ratios).
# ASSUMPTION: Any class whose name contains "Upland" is treated as upland.
# Update this set if your landcover legend uses different terminology.
UPLAND_CLASSES = {"Open Upland", "Treed Upland", "Forested Upland"}

# Output directory (relative to repo root)
OUTPUT_DIR = Path("../outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("All imports OK.")

## 2 · Feature Preparation

In [ ]:
def prepare_features(df: pl.DataFrame) -> pd.DataFrame:
    """
    Convert the Polars pixel DataFrame to a pandas DataFrame ready for XGBoost.

    Key transformations
    -------------------
    - Fire-weather columns (DC.mean etc.) are stored as strings in the source
      parquet; we cast them to float64.
    - lc_class_name is encoded as pd.Categorical so that XGBoost's
      enable_categorical=True pathway handles it natively (no one-hot encoding
      needed, no ordinal assumption imposed).
    - Only the columns needed for modelling are retained.
    """
    # Cast weather columns from string → float64
    cast_exprs = [pl.col(c).cast(pl.Float64) for c in WEATHER_COLS]

    pdf = (
        df
        .with_columns(cast_exprs)
        .select([TARGET_COL, LC_COL] + WEATHER_COLS + COORD_COLS)
        .to_pandas()
    )

    # Categorical dtype is required for XGBoost's enable_categorical=True
    pdf[LC_COL] = pd.Categorical(pdf[LC_COL])

    return pdf


pdf = prepare_features(hudson)

# ── Quick EDA ──────────────────────────────────────────────────────────────────
print(f"Feature matrix : {pdf.shape[0]:,} rows × {pdf.shape[1]} columns")
print(f"Burn rate      : {pdf[TARGET_COL].mean():.3%}")
print(f"Missing values : {pdf.isnull().sum().sum():,}")
print()
print("Landcover class breakdown (sorted by observed burn rate):")
class_stats = (
    pdf.groupby(LC_COL, observed=True)
    .agg(
        n_pixels   = (TARGET_COL, "count"),
        obs_burn_rate = (TARGET_COL, "mean"),
    )
    .sort_values("obs_burn_rate", ascending=False)
    .reset_index()
)
class_stats["pct_landscape"] = class_stats["n_pixels"] / len(pdf) * 100
print(class_stats.to_string(index=False, float_format=lambda x: f"{x:.3f}"))

## 3 · Spatial Block Cross-Validation

KMeans is run on pixel coordinates to create `N_BLOCKS` geographically contiguous
folds. Pixels in a test fold are spatially separated from their training pixels,
preventing autocorrelation leakage.

**Assumptions flagged here:**
- KMeans uses Euclidean distance in lon/lat space. This slightly underestimates
  true east-west distances at ~50–57°N (~20% compression). Projecting to UTM
  would be more exact but is not critical for fold assignment.
- `N_BLOCKS = 10` is a pragmatic choice. More blocks → smaller test sets, higher
  variance per fold; fewer blocks → larger test sets but fewer CV replicates for
  the bootstrap CI.

In [ ]:
N_BLOCKS = 10


def make_spatial_blocks(
    coords: np.ndarray,
    n_blocks: int,
    random_state: int = 42,
    n_fit: int = 500_000,
) -> np.ndarray:
    """
    Assign each pixel to one of `n_blocks` geographic clusters via KMeans.

    For large datasets we fit KMeans on a random subsample (`n_fit` pixels)
    then predict block labels for all pixels. This is equivalent to fitting on
    the full dataset because KMeans centroids converge well at n_fit ≫ n_clusters.

    Parameters
    ----------
    coords : (n_pixels, 2) array of [x, y] coordinates
    n_blocks : number of spatial blocks (= CV folds)
    random_state : reproducibility seed
    n_fit : number of points used to fit KMeans (subsampled for speed)

    Returns
    -------
    block_labels : (n_pixels,) int array, values in 0..n_blocks-1
    """
    n_fit = min(n_fit, len(coords))
    idx_fit = np.random.default_rng(random_state).choice(
        len(coords), n_fit, replace=False
    )
    km = KMeans(n_clusters=n_blocks, random_state=random_state, n_init=10)
    km.fit(coords[idx_fit])
    return km.predict(coords)


coords = pdf[COORD_COLS].values
block_labels = make_spatial_blocks(coords, N_BLOCKS, random_state=RANDOM_SEED)
pdf["block"] = block_labels  # store on the feature df for later use

# ── Block summary ──────────────────────────────────────────────────────────────
print(f"{'Block':>6}  {'N pixels':>11}  {'Burn rate':>10}  {'% of data':>10}")
print("-" * 44)
y_arr = pdf[TARGET_COL].values
for b in range(N_BLOCKS):
    mask = block_labels == b
    n    = mask.sum()
    br   = y_arr[mask].mean()
    print(f"  {b:3d}  {n:>11,}  {br:>10.3%}  {n/len(pdf):>10.2%}")

In [ ]:
# ─── Visualise spatial blocks ──────────────────────────────────────────────────
# Plot a subsample of pixels coloured by block ID to confirm geographic
# contiguity. Each block should form a spatially coherent region.

n_plot  = 300_000
idx_plt = rng.choice(len(pdf), n_plot, replace=False)

fig, ax = plt.subplots(figsize=(11, 7))
sc = ax.scatter(
    pdf["x"].values[idx_plt],
    pdf["y"].values[idx_plt],
    c=block_labels[idx_plt],
    cmap="tab10",
    s=0.4,
    alpha=0.5,
    rasterized=True,
)
cbar = plt.colorbar(sc, ax=ax, label="Block ID")
cbar.set_ticks(range(N_BLOCKS))
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_title(f"Spatial CV blocks (KMeans, k={N_BLOCKS}) — Hudson Plain\n"
             f"n={n_plot:,} pixels shown")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "spatial_blocks.png", dpi=150, bbox_inches="tight")
plt.show()

## 4 · Bayesian Hyperparameter Tuning (Optuna)

We run `N_TRIALS` Optuna trials using the TPE (Tree-structured Parzen Estimator)
sampler. For speed, tuning uses a stratified subsample and only `TUNE_CV_FOLDS`
spatial blocks as the inner CV.

**Assumptions flagged here:**
- `scale_pos_weight` is derived from the training data, not tuned, because it
  encodes the actual class imbalance rather than being a free hyperparameter.
  Tuning it would allow the model to ignore imbalance at the cost of calibration.
- `TUNE_N = 2_000_000` pixels is large enough to represent the landscape but
  keeps tuning tractable (~minutes rather than hours per trial). Increase if
  you have more compute budget.
- `TUNE_CV_FOLDS = 3` (subset of the 10 blocks) is used inside Optuna for speed.
  The full 10-fold OOF run uses the final best params.

In [ ]:
TUNE_N        = 2_000_000   # pixels to use for HP search
N_TRIALS      = 50          # Optuna trials
TUNE_CV_FOLDS = 3           # inner spatial CV folds (speed trade-off)

# ── Stratified subsample preserving burn rate ──────────────────────────────────
y_full       = pdf[TARGET_COL].values
idx_burned   = np.where(y_full == 1)[0]
idx_unburned = np.where(y_full == 0)[0]

n_burn_tune   = min(len(idx_burned),   int(TUNE_N * y_full.mean()))
n_unburn_tune = min(len(idx_unburned), TUNE_N - n_burn_tune)
tune_idx = np.concatenate([
    rng.choice(idx_burned,   n_burn_tune,   replace=False),
    rng.choice(idx_unburned, n_unburn_tune, replace=False),
])
rng.shuffle(tune_idx)

X_tune     = pdf[FEATURE_COLS].iloc[tune_idx].reset_index(drop=True)
y_tune     = y_full[tune_idx]
blk_tune   = block_labels[tune_idx]

# Build inner CV folds (first TUNE_CV_FOLDS block IDs)
inner_block_ids = sorted(np.unique(blk_tune))[:TUNE_CV_FOLDS]
inner_cv = [
    (np.where(blk_tune != b)[0], np.where(blk_tune == b)[0])
    for b in inner_block_ids
]

spw_tune = (y_tune == 0).sum() / max((y_tune == 1).sum(), 1)
print(f"Tuning sample  : {len(y_tune):,} pixels  (burn rate {y_tune.mean():.3%})")
print(f"scale_pos_weight: {spw_tune:.2f}")
print(f"Running {N_TRIALS} Optuna trials with {TUNE_CV_FOLDS}-fold inner spatial CV …")


def objective(trial: optuna.Trial) -> float:
    """Maximise mean OOF AUC-ROC across inner spatial folds."""
    params = {
        "n_estimators":     trial.suggest_int("n_estimators", 200, 1000),
        "max_depth":        trial.suggest_int("max_depth", 3, 9),
        "learning_rate":    trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 20),
        "subsample":        trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "reg_alpha":        trial.suggest_float("reg_alpha", 1e-4, 10.0, log=True),
        "reg_lambda":       trial.suggest_float("reg_lambda", 1e-4, 10.0, log=True),
        # Fixed settings
        "enable_categorical": True,
        "tree_method":        "hist",
        "eval_metric":        "auc",
        "random_state":       RANDOM_SEED,
        "n_jobs":             -1,
    }

    fold_aucs = []
    for train_idx, val_idx in inner_cv:
        X_tr, X_val = X_tune.iloc[train_idx], X_tune.iloc[val_idx]
        y_tr, y_val = y_tune[train_idx], y_tune[val_idx]

        # Recompute spw per fold so it reflects actual training-fold imbalance
        fold_spw = (y_tr == 0).sum() / max((y_tr == 1).sum(), 1)

        clf = xgb.XGBClassifier(**params, scale_pos_weight=fold_spw)
        clf.fit(X_tr, y_tr, verbose=False)
        prob = clf.predict_proba(X_val)[:, 1]
        fold_aucs.append(roc_auc_score(y_val, prob))

    return float(np.mean(fold_aucs))


study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=RANDOM_SEED),
)
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)

# ── Best params ────────────────────────────────────────────────────────────────
best_params = study.best_params.copy()
best_params.update({
    "enable_categorical": True,
    "tree_method":        "hist",
    "eval_metric":        "auc",
    "random_state":       RANDOM_SEED,
    "n_jobs":             -1,
})

print(f"\nBest inner-CV AUC : {study.best_value:.4f}")
print("Best hyperparameters:")
for k, v in best_params.items():
    print(f"  {k:22s}: {v}")

## 5 · Out-of-Fold Predictions

One XGBoost model is trained per spatial block (training on all *other* blocks,
predicting on the held-out block). Concatenating the predictions gives an
unbiased burn-probability estimate for every pixel — the OOF probabilities are
never informed by their own training labels.

In [ ]:
X_full    = pdf[FEATURE_COLS]
y_full    = pdf[TARGET_COL].values
oof_proba = np.full(len(y_full), np.nan, dtype=np.float32)
fold_models: dict[int, xgb.XGBClassifier] = {}

print(f"Running {N_BLOCKS}-fold spatial CV on {len(y_full):,} pixels …")
print(f"{'Fold':>5}  {'Test N':>11}  {'Burn rate':>10}  {'AUC':>8}")
print("-" * 42)

for fold_id in range(N_BLOCKS):
    train_mask = block_labels != fold_id
    test_mask  = block_labels == fold_id

    X_tr, X_te = X_full[train_mask], X_full[test_mask]
    y_tr, y_te = y_full[train_mask], y_full[test_mask]

    # Fold-specific scale_pos_weight (avoids leaking test-fold class counts)
    fold_spw = (y_tr == 0).sum() / max((y_tr == 1).sum(), 1)

    clf = xgb.XGBClassifier(
        **{k: v for k, v in best_params.items() if k != "scale_pos_weight"},
        scale_pos_weight=fold_spw,
    )
    clf.fit(X_tr, y_tr, verbose=False)

    fold_prob = clf.predict_proba(X_te)[:, 1]
    oof_proba[test_mask] = fold_prob
    fold_models[fold_id] = clf

    fold_auc = roc_auc_score(y_te, fold_prob)
    print(f"  {fold_id:3d}  {test_mask.sum():>11,}  {y_te.mean():>10.3%}  {fold_auc:>8.4f}")

assert not np.isnan(oof_proba).any(), "Some pixels have no OOF prediction!"
overall_auc = roc_auc_score(y_full, oof_proba)
print("-" * 42)
print(f"Overall OOF AUC : {overall_auc:.4f}")

# Persist OOF predictions on the main DataFrame
pdf["oof_proba"] = oof_proba

## 6 · Calibration Check

Before using OOF probabilities as burn-probability estimates, we verify they are
well-calibrated (i.e., a predicted probability of 0.3 truly corresponds to ~30%
of those pixels burning). Poor calibration would bias the log odds selectivity
estimates.

**Key metrics:**
- **Brier score** — mean squared error of probability predictions; the null
  model (always predict the base rate) scores `p × (1−p)`.
- **ECE** — Expected Calibration Error; the area-weighted mean absolute
  deviation between predicted and observed probability in quantile bins.

In [ ]:
def expected_calibration_error(
    y_true: np.ndarray,
    y_prob: np.ndarray,
    n_bins: int = 15,
) -> float:
    """
    Expected Calibration Error (ECE) using equal-frequency (quantile) bins.

    ECE = Σ_b  (n_b / N) × |mean_predicted_b − mean_observed_b|

    Quantile binning is used because it avoids empty bins when the predicted
    probability distribution is skewed (as is typical for rare-event classifiers).
    """
    quantiles = np.linspace(0, 1, n_bins + 1)
    bin_edges = np.quantile(y_prob, quantiles)
    bin_edges[-1] += 1e-8  # ensure the max value is included

    ece = 0.0
    n   = len(y_true)
    for lo, hi in zip(bin_edges[:-1], bin_edges[1:]):
        mask = (y_prob >= lo) & (y_prob < hi)
        if mask.sum() == 0:
            continue
        ece += mask.sum() / n * abs(y_true[mask].mean() - y_prob[mask].mean())
    return ece


base_rate   = y_full.mean()
brier       = brier_score_loss(y_full, oof_proba)
brier_null  = base_rate * (1 - base_rate)
ece         = expected_calibration_error(y_full, oof_proba)

print(f"Brier score (model) : {brier:.4f}")
print(f"Brier score (null)  : {brier_null:.4f}  (predict base rate = {base_rate:.3%})")
print(f"Brier skill score   : {1 - brier / brier_null:.3f}  (higher = better)")
print(f"ECE                 : {ece:.4f}  (0 = perfect calibration)")
print()
if ece > 0.02:
    print("⚠  ECE > 0.02 — consider Platt scaling or isotonic regression before "
          "interpreting probabilities as absolute burn risks.")
else:
    print("✓  ECE ≤ 0.02 — probabilities appear well-calibrated for selectivity analysis.")

# ── Reliability diagram ────────────────────────────────────────────────────────
frac_pos, mean_pred = calibration_curve(y_full, oof_proba, n_bins=20, strategy="quantile")

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
ax.plot([0, 1], [0, 1], "k--", lw=1, label="Perfect calibration")
ax.plot(mean_pred, frac_pos, "o-", color="steelblue", lw=2, ms=5, label="XGBoost OOF")
ax.set_xlabel("Mean predicted probability")
ax.set_ylabel("Observed burn rate")
ax.set_title("Reliability diagram")
ax.legend()
ax.text(
    0.05, 0.93,
    f"ECE = {ece:.4f}\nBrier = {brier:.4f}\nSkill = {1-brier/brier_null:.3f}",
    transform=ax.transAxes, va="top",
    bbox=dict(boxstyle="round", fc="white", alpha=0.85),
)

ax = axes[1]
ax.hist(oof_proba[y_full == 0], bins=60, alpha=0.6, density=True,
        color="steelblue", label="Unburned (0)")
ax.hist(oof_proba[y_full == 1], bins=60, alpha=0.6, density=True,
        color="firebrick",  label="Burned (1)")
ax.set_xlabel("Predicted burn probability")
ax.set_ylabel("Density")
ax.set_title("OOF probability distribution by label")
ax.legend()

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "calibration.png", dpi=150, bbox_inches="tight")
plt.show()

## 7 · Log Odds Selectivity

**Method:**

1. For each landcover class, compute its *model-adjusted* mean burn probability
   as the mean OOF predicted probability across all pixels of that class.
   Using predicted probabilities (rather than raw burn counts) controls for
   differences in fire weather and season across classes.

2. Express each class's burn probability as a **log odds ratio (LOR)** relative to
   (A) the landscape-wide mean probability and (B) the pooled upland mean probability.

   `LOR_class_vs_ref = logit(p_class) − logit(p_ref)`

   - LOR > 0 → class burns *more* than the reference
   - LOR < 0 → class is selectively *avoided* by fire

3. **Confidence intervals** are obtained by bootstrapping *folds* with replacement
   (`N_BOOT` iterations). Pixels within a fold are spatially correlated and are not
   exchangeable — fold-level bootstrap treats each fold as a geographic replicate.

**Assumptions flagged here:**
- Upland = pooled mean across Open Upland, Treed Upland, Forested Upland.
  If these classes behave heterogeneously you may want separate upland references.
- The `logit` function is clipped at [1e-6, 1−1e-6] to avoid ±∞ for
  classes with near-zero or near-one burn rates.

In [ ]:
def logit(p: float | np.ndarray) -> float | np.ndarray:
    """Log-odds of p, clipped to avoid ±∞."""
    p = np.clip(p, 1e-6, 1.0 - 1e-6)
    return np.log(p / (1.0 - p))


def compute_selectivity(
    df: pd.DataFrame,
    lc_col: str,
    proba_col: str,
    target_col: str,
    exclude_classes: set,
    upland_classes: set,
) -> tuple[pd.DataFrame, float, float]:
    """
    Compute per-class log odds selectivity from predicted probabilities.

    Parameters
    ----------
    df             : pixel-level DataFrame with OOF probabilities
    lc_col         : column name for landcover class (string)
    proba_col      : column name for OOF predicted burn probability
    target_col     : column name for observed burn label (0/1)
    exclude_classes: classes to omit (Water, None, etc.)
    upland_classes : classes defining the upland reference group

    Returns
    -------
    sel_df          : per-class selectivity DataFrame
    p_landscape     : landscape mean predicted probability
    p_upland        : upland mean predicted probability
    """
    # Drop excluded classes
    mask = ~df[lc_col].isin(exclude_classes) & df[lc_col].notna()
    sub  = df[mask].copy()

    p_landscape  = sub[proba_col].mean()
    lo_landscape = logit(p_landscape)

    upland_mask = sub[lc_col].isin(upland_classes)
    p_upland    = sub.loc[upland_mask, proba_col].mean()
    lo_upland   = logit(p_upland)

    rows = []
    for cls, grp in sub.groupby(lc_col, observed=True):
        p_cls = grp[proba_col].mean()
        lo_cls = logit(p_cls)
        rows.append({
            "lc_class_name":     cls,
            "n_pixels":          len(grp),
            "mean_proba":        p_cls,
            "obs_burn_rate":     grp[target_col].mean(),
            "n_burned":          int(grp[target_col].sum()),
            "lor_vs_landscape":  float(lo_cls - lo_landscape),
            "lor_vs_upland":     float(lo_cls - lo_upland),
            "is_upland":         cls in upland_classes,
        })

    sel_df = (
        pd.DataFrame(rows)
        .sort_values("lor_vs_landscape", ascending=False)
        .reset_index(drop=True)
    )
    return sel_df, p_landscape, p_upland


sel_df, p_land, p_up = compute_selectivity(
    pdf, LC_COL, "oof_proba", TARGET_COL,
    EXCLUDE_CLASSES, UPLAND_CLASSES,
)

print(f"Landscape mean predicted burn probability : {p_land:.3%}")
print(f"Upland    mean predicted burn probability : {p_up:.3%}")
print()
print(sel_df[[
    "lc_class_name", "n_pixels", "obs_burn_rate", "mean_proba",
    "lor_vs_landscape", "lor_vs_upland", "is_upland"
]].to_string(index=False, float_format=lambda x: f"{x:+.3f}"))

In [ ]:
N_BOOT = 1_000   # bootstrap iterations


def bootstrap_selectivity_ci(
    df: pd.DataFrame,
    lc_col: str,
    proba_col: str,
    block_col: str,
    exclude_classes: set,
    upland_classes: set,
    n_boot: int = 1_000,
    seed: int = 42,
    alpha: float = 0.05,
) -> pd.DataFrame:
    """
    Bootstrap 95% CIs for LOR vs landscape and LOR vs upland by resampling
    *spatial folds* with replacement.

    Fold-level bootstrap (rather than pixel-level) is used because pixels
    within a fold are spatially autocorrelated; they are not exchangeable units.
    Each fold is treated as one independent geographic replicate.

    Parameters
    ----------
    df              : pixel DataFrame with columns lc_col, proba_col, block_col
    lc_col          : landcover class column
    proba_col       : OOF predicted probability column
    block_col       : spatial block / fold assignment column
    exclude_classes : classes to skip
    upland_classes  : classes forming the upland reference group
    n_boot          : number of bootstrap iterations
    seed            : RNG seed
    alpha           : two-tailed CI level (default 0.05 → 95% CI)

    Returns
    -------
    ci_df : DataFrame indexed by lc_class_name with LOR mean ± CI columns
    """
    mask = ~df[lc_col].isin(exclude_classes) & df[lc_col].notna()
    sub  = df[mask].copy()

    blocks     = sub[block_col].unique()
    n_blocks   = len(blocks)
    boot_rng   = np.random.default_rng(seed)

    # Pre-split by block for fast iteration
    block_dfs = {b: sub[sub[block_col] == b] for b in blocks}
    all_classes = sub[lc_col].unique().tolist()

    # Storage: lor_land_boot[cls] = list of bootstrap LOR values
    lor_land = {cls: [] for cls in all_classes}
    lor_up   = {cls: [] for cls in all_classes}

    for _ in range(n_boot):
        sampled = boot_rng.choice(blocks, size=n_blocks, replace=True)
        boot_df = pd.concat([block_dfs[b] for b in sampled], ignore_index=True)

        p_land_b  = boot_df[proba_col].mean()
        lo_land_b = logit(p_land_b)

        up_mask   = boot_df[lc_col].isin(upland_classes)
        p_up_b    = boot_df.loc[up_mask, proba_col].mean()
        lo_up_b   = logit(p_up_b)

        for cls, grp in boot_df.groupby(lc_col, observed=True):
            if cls not in lor_land:
                continue
            lo_cls = logit(grp[proba_col].mean())
            lor_land[cls].append(float(lo_cls - lo_land_b))
            lor_up[cls].append(float(lo_cls - lo_up_b))

    lo_q, hi_q = alpha / 2 * 100, (1 - alpha / 2) * 100
    ci_rows = []
    for cls in all_classes:
        ll = np.array(lor_land[cls])
        lu = np.array(lor_up[cls])
        ci_rows.append({
            "lc_class_name":          cls,
            "lor_vs_landscape_mean":  ll.mean(),
            "lor_vs_landscape_lo":    float(np.percentile(ll, lo_q)),
            "lor_vs_landscape_hi":    float(np.percentile(ll, hi_q)),
            "lor_vs_upland_mean":     lu.mean(),
            "lor_vs_upland_lo":       float(np.percentile(lu, lo_q)),
            "lor_vs_upland_hi":       float(np.percentile(lu, hi_q)),
        })

    return pd.DataFrame(ci_rows)


print(f"Bootstrapping {N_BOOT} fold-resamples …")
ci_df = bootstrap_selectivity_ci(
    pdf, LC_COL, "oof_proba", "block",
    EXCLUDE_CLASSES, UPLAND_CLASSES,
    n_boot=N_BOOT, seed=RANDOM_SEED,
)

# Merge point estimates + CIs into one results table
results = (
    sel_df
    .merge(ci_df, on="lc_class_name")
    .sort_values("lor_vs_landscape_mean", ascending=False)
    .reset_index(drop=True)
)

print(f"\nSelectivity table ({len(results)} classes, ranked by LOR vs landscape):")
print(results[[
    "lc_class_name", "n_pixels", "obs_burn_rate", "mean_proba",
    "lor_vs_landscape_mean", "lor_vs_landscape_lo", "lor_vs_landscape_hi",
    "lor_vs_upland_mean",    "lor_vs_upland_lo",    "lor_vs_upland_hi",
    "is_upland",
]].to_string(index=False, float_format=lambda x: f"{x:+.3f}"))

In [ ]:
# ─── Selectivity plot (ranked forest plot) ────────────────────────────────────
plot_df = results.sort_values("lor_vs_landscape_mean")
colors  = ["#c0392b" if u else "#2980b9" for u in plot_df["is_upland"]]

fig, axes = plt.subplots(1, 2, figsize=(15, 0.55 * len(plot_df) + 2), sharey=True)

specs = [
    ("lor_vs_landscape_mean", "lor_vs_landscape_lo", "lor_vs_landscape_hi",
     "Landscape mean"),
    ("lor_vs_upland_mean",    "lor_vs_upland_lo",    "lor_vs_upland_hi",
     "Upland"),
]

for ax, (lor_col, lo_col, hi_col, ref_label) in zip(axes, specs):
    y_pos = np.arange(len(plot_df))
    lor   = plot_df[lor_col].values
    lo    = plot_df[lo_col].values
    hi    = plot_df[hi_col].values

    ax.barh(y_pos, lor, color=colors, alpha=0.75, height=0.6)
    ax.errorbar(
        lor, y_pos,
        xerr=[lor - lo, hi - lor],
        fmt="none", color="black", capsize=3, linewidth=1,
    )
    ax.axvline(0, color="black", linewidth=1.5, linestyle="--")
    ax.set_yticks(y_pos)
    ax.set_yticklabels(plot_df["lc_class_name"], fontsize=9)
    ax.set_xlabel("Log Odds Ratio", fontsize=10)
    ax.set_title(f"Selectivity vs. {ref_label}", fontsize=11)
    ax.grid(axis="x", alpha=0.3)

legend_handles = [
    mpatches.Patch(facecolor="#c0392b", alpha=0.75, label="Upland"),
    mpatches.Patch(facecolor="#2980b9", alpha=0.75, label="Peatland / other"),
]
axes[1].legend(handles=legend_handles, loc="lower right", fontsize=9)

fig.suptitle(
    "Fire selectivity — Hudson Plain, 2023 Canadian wildfires\n"
    "bars = model-adjusted LOR (OOF probabilities); whiskers = 95% CI (fold bootstrap)",
    fontsize=11,
)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "selectivity.png", dpi=150, bbox_inches="tight")
plt.show()

## 8 · SHAP Analysis

We train a **final model on all pixels** (no holdout) using the tuned
hyperparameters. SHAP values from this model decompose each individual
prediction into additive feature contributions in log-odds units.

**Interpretation guide:**
- SHAP value for pixel *i* and feature *j* = the contribution of feature *j*
  to pixel *i*'s predicted log-odds, relative to the global base value
  (= logit of the mean predicted probability across all training data).
- Positive SHAP → feature value pushes the prediction toward *burned*.
- Negative SHAP → feature value pushes the prediction toward *unburned*.

**Variance decomposition:**
We estimate how much of the *total variance in predicted burn probability*
is attributable to each feature by comparing Var(SHAP_j) across features.
Note: SHAP values are additive in log-odds space, so Var(SHAP_j) is a valid
decomposition of model variance. This is **model variance**, not observed
burn variance — it answers "how much does landcover vs. fire weather drive
the model's predictions?"

**Assumptions flagged here:**
- SHAP uses `tree_path_dependent` (default for `TreeExplainer`), which is an
  exact, fast method that conditions on the tree structure. It is appropriate
  for measuring feature importance within this model, but gives *correlated-
  feature* SHAP values (not marginal/causal). Use `interventional=True` with
  a background dataset for marginal SHAP.
- SHAP is computed on `SHAP_N = 50_000` pixels stratified by landcover class.
  All classes are guaranteed representation.

In [ ]:
SHAP_N = 50_000

# ── Train final model on all data ─────────────────────────────────────────────
spw_final   = (y_full == 0).sum() / max((y_full == 1).sum(), 1)
final_model = xgb.XGBClassifier(
    **{k: v for k, v in best_params.items() if k != "scale_pos_weight"},
    scale_pos_weight=spw_final,
)
final_model.fit(X_full, y_full, verbose=False)
print("Final model trained on all data.")

# ── Stratified SHAP subsample ─────────────────────────────────────────────────
# Sample proportionally from each landcover class so rare classes are represented
n_classes       = pdf[LC_COL].nunique()
per_class_limit = max(1, SHAP_N // n_classes)

shap_parts = []
for cls, grp in pdf.groupby(LC_COL, observed=True):
    sample_n = min(len(grp), per_class_limit)
    shap_parts.append(
        grp.sample(sample_n, random_state=RANDOM_SEED)
    )
shap_df  = pd.concat(shap_parts).sample(frac=1, random_state=RANDOM_SEED)  # shuffle
shap_df  = shap_df.head(SHAP_N)  # cap at SHAP_N if slightly over
X_shap   = shap_df[FEATURE_COLS]

print(f"Computing SHAP on {len(X_shap):,} pixels "
      f"({pdf[LC_COL].nunique()} classes, up to {per_class_limit} each) …")

explainer   = shap.TreeExplainer(final_model)
shap_values = explainer(X_shap)   # Explanation object (.values, .base_values, .data)
shap_arr    = shap_values.values  # (n_shap, n_features) in log-odds space

print("SHAP computation complete.")

In [ ]:
# ─── SHAP beeswarm plot ────────────────────────────────────────────────────────
# Shows the distribution of SHAP values for each feature across the SHAP sample.
# Each dot is one pixel; colour encodes the feature value (red = high, blue = low).
# Width of the swarm shows how many pixels fall at each SHAP value.

fig, ax = plt.subplots(figsize=(10, 6))
shap.plots.beeswarm(shap_values, max_display=len(FEATURE_COLS), show=False, ax=ax)
ax.set_title(
    "SHAP beeswarm — feature contributions to burn log-odds\n"
    "(final model, all data; SHAP sample n=50,000)"
)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "shap_beeswarm.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ─── Landcover SHAP deep-dive ──────────────────────────────────────────────────
# Left panel : per-class SHAP boxplot for lc_class_name, ordered by median SHAP.
#   Classes with SHAP > 0 → landcover pushes the model toward burning.
#   Classes with SHAP < 0 → landcover damps burn probability.
# Right panel: variance decomposition — how much of the MODEL'S predicted-
#   probability variance is driven by each feature.

lc_idx    = list(FEATURE_COLS).index(LC_COL)
shap_lc   = shap_arr[:, lc_idx]          # SHAP values for lc_class_name only
lc_labels = X_shap[LC_COL].astype(str).values

# ── Variance decomposition ─────────────────────────────────────────────────────
# Var(Σ SHAP_j) ≈ Var of total model log-odds;
# Var(SHAP_j) for each feature captures that feature's independent contribution
# (this assumes low interaction, which XGBoost's additive decomposition supports)
var_per_feat  = shap_arr.var(axis=0)
var_total     = shap_arr.sum(axis=1).var()
pct_var       = var_per_feat / var_total * 100

feat_var_df = (
    pd.DataFrame({"feature": FEATURE_COLS, "pct_variance": pct_var})
    .sort_values("pct_variance", ascending=False)
    .reset_index(drop=True)
)

print("Variance of total SHAP (log-odds):", f"{var_total:.4f}")
print("\nFeature contributions to model variance (SHAP-based):")
for _, row in feat_var_df.iterrows():
    marker = " ← landcover" if row["feature"] == LC_COL else ""
    print(f"  {row['feature']:22s}: {row['pct_variance']:6.2f}%{marker}")

# ── Plot ───────────────────────────────────────────────────────────────────────
class_order = (
    pd.Series(shap_lc, index=lc_labels)
    .groupby(level=0)
    .median()
    .sort_values()
    .index.tolist()
)
box_data   = [shap_lc[lc_labels == cls] for cls in class_order]
box_colors = ["#c0392b" if cls in UPLAND_CLASSES else "#2980b9" for cls in class_order]

fig, axes = plt.subplots(1, 2, figsize=(16, max(6, 0.45 * len(class_order) + 2)))

# Left: per-class SHAP boxplot
ax = axes[0]
bp = ax.boxplot(
    box_data, vert=False, patch_artist=True, labels=class_order,
    flierprops=dict(marker=".", markersize=2, alpha=0.3),
)
for patch, col in zip(bp["boxes"], box_colors):
    patch.set_facecolor(col)
    patch.set_alpha(0.75)
ax.axvline(0, color="black", lw=1.5, ls="--")
ax.set_xlabel("SHAP value (contribution to burn log-odds)", fontsize=10)
ax.set_title(f"SHAP values for {LC_COL}\n(positive = more likely to burn)", fontsize=10)
legend_handles = [
    mpatches.Patch(facecolor="#c0392b", alpha=0.75, label="Upland"),
    mpatches.Patch(facecolor="#2980b9", alpha=0.75, label="Peatland / other"),
]
ax.legend(handles=legend_handles, fontsize=9)
ax.grid(axis="x", alpha=0.3)

# Right: variance decomposition bar chart
ax = axes[1]
bar_cols = ["#c0392b" if f == LC_COL else "#7f8c8d" for f in feat_var_df["feature"]]
ax.barh(
    feat_var_df["feature"].str.replace(".mean", "", regex=False),
    feat_var_df["pct_variance"],
    color=bar_cols,
    alpha=0.8,
)
ax.set_xlabel("% of total SHAP variance", fontsize=10)
ax.set_title("Feature contribution to\nmodel burn-probability variance", fontsize=10)
ax.grid(axis="x", alpha=0.3)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "shap_landcover.png", dpi=150, bbox_inches="tight")
plt.show()

## 9 · Summary & Final Results Table

In [ ]:
# ─── Formatted summary table ───────────────────────────────────────────────────
def fmt_ci(mean: float, lo: float, hi: float) -> str:
    return f"{mean:+.2f} [{lo:+.2f}, {hi:+.2f}]"


summary = results.copy()
summary["LOR vs landscape (95% CI)"] = [
    fmt_ci(r.lor_vs_landscape_mean, r.lor_vs_landscape_lo, r.lor_vs_landscape_hi)
    for r in summary.itertuples()
]
summary["LOR vs upland (95% CI)"] = [
    fmt_ci(r.lor_vs_upland_mean, r.lor_vs_upland_lo, r.lor_vs_upland_hi)
    for r in summary.itertuples()
]
summary["obs_burn_rate_%"] = (summary["obs_burn_rate"] * 100).round(2)
summary["mean_proba_%"]    = (summary["mean_proba"] * 100).round(2)

cols = [
    "lc_class_name", "n_pixels", "obs_burn_rate_%", "mean_proba_%",
    "LOR vs landscape (95% CI)", "LOR vs upland (95% CI)", "is_upland",
]
display_df = summary[cols].sort_values("mean_proba_%", ascending=False)

print("=" * 110)
print("FINAL SELECTIVITY TABLE  |  Hudson Plain 2023 Canadian wildfires")
print("=" * 110)
print(display_df.to_string(index=False))
print()
print("─" * 110)
print(f"Landscape mean predicted burn probability : {p_land:.3%}")
print(f"Upland    mean predicted burn probability : {p_up:.3%}")
print(f"Overall OOF AUC                           : {overall_auc:.4f}")
print()
print("INTERPRETATION")
print("  LOR vs landscape > 0 → class burns MORE than the average pixel")
print("  LOR vs landscape < 0 → class is selectively avoided by fire")
print("  LOR vs upland    < 0 → class burns LESS than upland (peatland selectivity)")
print()
print("OUTPUTS WRITTEN TO")
for f in sorted(OUTPUT_DIR.glob("*.png")):
    print(f"  {f}")

# Save results table to CSV
out_csv = OUTPUT_DIR / "selectivity_results.csv"
summary[cols].to_csv(out_csv, index=False)
print(f"  {out_csv}")